**0.0 Data Extraction (non-rerunnable)**

This part extracts the comments from the Reddit threats by using their URL link. Then the data is compiled into one overall dataset. To run this code, one must add their personal Reddit ID information, which would extract the latest version of the discussion thread.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q praw

import os
import praw
import pandas as pd
import time


REDDIT_CLIENT_ID = os.getenv("REDDIT_CLIENT_ID", "").strip()
REDDIT_CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET", "").strip()
REDDIT_USER_AGENT = os.getenv(
    "REDDIT_USER_AGENT",
    "reddit-extraction (by u/your_username)"
).strip()

if not REDDIT_CLIENT_ID or not REDDIT_CLIENT_SECRET:
    raise RuntimeError(
        "❌ Reddit API credentials not found.\n\n"
        "Please set the following environment variables in Colab:\n\n"
        '  os.environ["REDDIT_CLIENT_ID"] = "YOUR_CLIENT_ID"\n'
        '  os.environ["REDDIT_CLIENT_SECRET"] = "YOUR_CLIENT_SECRET"\n'
        '  os.environ["REDDIT_USER_AGENT"] = "YOUR_USER_AGENT"\n\n'
        "Then re-run this cell."
    )


reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT,
    check_for_async=False
)


try:
    _ = reddit.user.me()
    print("✅ Reddit API authentication successful.")
except Exception as e:
    print("⚠️ Authentication check could not confirm user identity (read-only access may still work).")
    print("Details:", repr(e))


thread_urls = [
    "https://www.reddit.com/r/VictoriasSecret/comments/1o7rlmq/fashion_show_2025_opinions/",
    "https://www.reddit.com/r/popculturechat/comments/1o7t451/the_victorias_secret_fashion_show_2025/",
    "https://www.reddit.com/r/VictoriasSecret/comments/1o7q8oa/discuss_the_vs_fashion_show_under_here/"
]


rows = []

for url in thread_urls:
    print(f"\n🔍 Processing thread:\n{url}")
    submission = reddit.submission(url=url)

    # Force fetch
    _ = submission.title
    print(f"Reddit reports ~{submission.num_comments} comments")

    submission.comments.replace_more(limit=None)
    comments = submission.comments.list()

    unique_comments = {c.id: c for c in comments}
    print(f"Collected {len(unique_comments)} unique comments")

    for c in unique_comments.values():
        rows.append({
            "thread_url": url,
            "thread_id": submission.id,
            "thread_title": submission.title,
            "comment_id": c.id,
            "parent_id": c.parent_id,
            "author": str(c.author) if c.author is not None else "[deleted]",
            "body": c.body,
            "score": c.score,
            "created_utc": c.created_utc
        })

    # Be polite to Reddit
    time.sleep(1)


df_raw = pd.DataFrame(rows)

if df_raw.empty:
    print("\n⚠️ No comments collected. Check thread URLs and credentials.")
else:
    print(f"\n✅ Total unique comments collected: {df_raw['comment_id'].nunique():,}")
    print(df_raw.head(3))

output_path = "/content/drive/MyDrive/reddit_extracted_raw_comments.csv"
df_raw.to_csv(output_path, index=False, encoding="utf-8")

print(f"\n💾 Raw Reddit comments saved to Google Drive at:\n{output_path}")


**1.0 Data Preparation (non-rerunnable, privacy-restriced)**

This part reviews the data and prepares it by removing any deleted comments, adding a stable index and anonymising usernames for privacy reasons. The stable index ensures consistency during the thematic coding, especially considering the dataset may be reloaded which causes default indexes to change.

For privacy reasons, this part of the data analysis is not rerunable in Github, as the file contains the original authornames. For rerunning the code, please start from step **1.1 GitHub-Ready Code**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd


raw_path_drive = "/content/drive/MyDrive/reddit_extracted_raw_comments.csv"
clean_path_drive = "/content/drive/MyDrive/reddit_cleaned_dataset.csv"

df = pd.read_csv(raw_path_drive)
print(f"Loaded raw dataset: {len(df):,} rows")

# Remove comments where body is "[deleted]" or "[removed]"
df["body"] = df["body"].astype(str)
df_clean = df[~df["body"].isin(["[deleted]", "[removed]"])].copy()
print(f"After removing [deleted]/[removed] bodies: {len(df_clean):,} rows")


if "author" not in df_clean.columns:
    raise ValueError("❌ Column 'author' not found in dataset")

author_series = df_clean["author"].astype(str).str.strip()


is_deleted_author = (
    df_clean["author"].isna() |
    author_series.eq("") |
    author_series.str.lower().isin(["none", "deleted", "[deleted]", "[removed]"])
)

valid_authors = author_series[~is_deleted_author].unique()
author_to_anon = {author: f"User_{i+1}" for i, author in enumerate(valid_authors)}

df_clean["author_anon"] = author_series.map(author_to_anon)
df_clean.loc[is_deleted_author, "author_anon"] = "user_deleted"

print(f"👤 Created 'author_anon' for {len(valid_authors):,} users.")
print("🕵️ Added 'author_anon' column")

# PRIVACY: drop original author column
df_clean.drop(columns=["author"], inplace=True)
print("🔒 Dropped identifying column: 'author'")

# Add stable index
df_clean = df_clean.reset_index(drop=True)

if "stable_index" not in df_clean.columns:
    df_clean["stable_index"] = df_clean.index
    print("📌 Added stable 'stable_index' column")


cols = df_clean.columns.tolist()
cols.insert(0, cols.pop(cols.index("stable_index")))
df_clean = df_clean[cols]

df_clean.to_csv(clean_path_drive, index=False, encoding="utf-8")
print(f"✅ Saved cleaned dataset to Drive: {clean_path_drive}")
print(f"📌 Final rows: {len(df_clean):,}")

print("\nFinal Reddit cleaned dataframe info:")
df_clean.info()


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/reddit_cleaned_dataset.csv"
df_new = pd.read_csv(file_path)

print(f"✅ Loaded {len(df_new):,} rows from {file_path}")

pd.set_option('display.max_colwidth', None)

df_new["body"].head()


**1.1 GitHub-Ready Code**

In [ ]:
import shutil
from pathlib import Path
import os

REPO = "data-analysis-Reddit"
REPO_URL = "https://github.com/salomevanzutphen/data-analysis-Reddit.git"

if Path(REPO).exists():
    shutil.rmtree(REPO)

!git clone {REPO_URL}
os.chdir(REPO)

!ls -lh data/dataset/reddit_cleaned_dataset.csv


In [ ]:
import pandas as pd

df_reddit = pd.read_csv("data/dataset/reddit_cleaned_dataset.csv")
print(df_reddit.shape)
df_reddit.head()


In [ ]:
from pathlib import Path
import pandas as pd

CANDIDATE_PATHS = [
    Path("data/dataset/reddit_cleaned_dataset.csv"),
    Path("../data/dataset/reddit_cleaned_dataset.csv"),
]

CLEAN_REDDIT_DATA = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if CLEAN_REDDIT_DATA is None:
    raise FileNotFoundError(
        "Could not find reddit_cleaned_dataset.csv.\n"
        "Expected it at data/dataset/reddit_cleaned_dataset.csv"
    )

df_reddit = pd.read_csv(CLEAN_REDDIT_DATA)
print(f"✅ Loaded {len(df_reddit):,} Reddit rows from: {CLEAN_REDDIT_DATA}")


**1.2 Reviewing Duplicate Comments**

This part includes reviewing the metrics of duplicate comments, to determine whether they are true or accidental duplications.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import display


df_reddit = pd.read_csv(CLEAN_REDDIT_DATA)
print(f"Loaded {len(df_reddit):,} Reddit rows")

text_col = "body"


df_reddit["created_utc_dt"] = pd.to_datetime(
    df_reddit["created_utc"], unit="s", utc=True, errors="coerce"
)


duplicate_counts = df_reddit[text_col].value_counts()
duplicate_texts = duplicate_counts[duplicate_counts > 1].index.tolist()

print(f"🔍 Found {len(duplicate_texts)} duplicated unique Reddit comments.")


duplicates_full = df_reddit[df_reddit[text_col].isin(duplicate_texts)].copy()

def summarize_group(g):

    g = g.sort_values("created_utc_dt")
    freq = len(g)

    rows = []
    for _, r in g.iterrows():
        rows.append(
            f"user={r.get('author_anon', '')}, "
            f"score={r.get('score', '')}, "
            f"createdAt={r.get('created_utc_dt', '')}"
        )

    return pd.Series({
        "frequency": freq,
        "instances": "\n".join(rows)
    })


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    summary_reddit = duplicates_full.groupby(text_col).apply(summarize_group).reset_index()


summary_reddit = summary_reddit.sort_values("frequency", ascending=False).reset_index(drop=True)


summary_reddit.index = summary_reddit.index + 1

# Display
pd.set_option("display.max_colwidth", None)

display(
    summary_reddit.style.set_properties(
        subset=[text_col],
        **{
            "white-space": "normal",
            "max-width": "300px",
        }
    ).set_properties(
        subset=["instances"],
        **{
            "white-space": "pre-wrap",
            "max-width": "1000px",
            "font-family": "monospace"
        }
    )
)


**2.0 Opinion Leaders**

The opinion leaders are identified using the benchmark of 75% of the total likecount/upvotes of the overall dataset. After knowing how many comments are in this subset, they are reviewed with their ranking, stable index, like count and text.

In [ ]:
import pandas as pd

df = pd.read_csv(CLEAN_REDDIT_DATA)

if "score" not in df.columns:
    raise ValueError("Column 'score' not found in df")


total_comments = len(df)
total_upvotes = df["score"].sum()


if total_comments == 0 or total_upvotes == 0:
    print("No comments or no upvotes in the dataset.")
else:

    target_upvotes_75 = 0.75 * total_upvotes

    df_sorted = df.sort_values("score", ascending=False).reset_index(drop=True)
    df_sorted["cum_upvotes"] = df_sorted["score"].cumsum()

    first_row_reaching_75 = df_sorted[df_sorted["cum_upvotes"] >= target_upvotes_75].index.min()
    comments_needed = int(first_row_reaching_75) + 1

    percent_of_dataset = (comments_needed / total_comments) * 100


    print(f"Total comments: {total_comments:,}")
    print(f"Total upvotes: {total_upvotes:,}")
    print(f"75% benchmark of total upvotes: {target_upvotes_75:,.0f}")
    print(f"Comments needed to reach 75% of upvotes: {comments_needed:,}")
    print(f"These comments represent {percent_of_dataset:.2f}% of the dataset.")


Unlike other platforms, on Reddit some comments have the exact same score of upvotes. This makes the ranking of the top 144 comments suspect to change. Based on the reloading of the dataset, the final list would change, as the system would randomly pick one of the few comments with the same ranking.  Therefore, an added internal ranking system is added based on an early advantage standard, which contributes to increased visibility. This means that apart from score, the date and timing of comments will be taken into account, to create a consistent stable rank.

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(CLEAN_REDDIT_DATA)

SCORE_COL = "score"
COMMENT_COL = "body"
STABLE_ID_COL = "stable_index"

TIME_COL_CANDIDATES = ["created_utc", "created", "createdAt", "timestamp", "time"]
TIME_COL = next((c for c in TIME_COL_CANDIDATES if c in df.columns), None)

if TIME_COL is None:
    raise ValueError(
        f"No timestamp column found. Tried {TIME_COL_CANDIDATES}. "
        f"Available columns: {list(df.columns)}"
    )

for col in [SCORE_COL, COMMENT_COL, STABLE_ID_COL, TIME_COL]:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found.")

df[COMMENT_COL] = df[COMMENT_COL].fillna("").astype(str)
df[SCORE_COL] = pd.to_numeric(df[SCORE_COL], errors="coerce").fillna(0).astype(int)

if "utc" in TIME_COL.lower():
    df["comment_time_utc"] = pd.to_datetime(
        df[TIME_COL], unit="s", errors="coerce", utc=True
    )
else:
    df["comment_time_utc"] = pd.to_datetime(
        df[TIME_COL], errors="coerce", utc=True
    )

df["comment_time_ams"] = df["comment_time_utc"].dt.tz_convert("Europe/Amsterdam")

df["comment_time_ams"] = df["comment_time_ams"].fillna(
    pd.Timestamp.max.tz_localize("Europe/Amsterdam")
)

opinionleaders_reddit = (
    df.sort_values(
        by=[SCORE_COL, "comment_time_ams", STABLE_ID_COL],
        ascending=[False, True, True],
        kind="mergesort"
    )
    .head(144)
    .copy()
)

opinionleaders_reddit.insert(0, "rank", range(1, len(opinionleaders_reddit) + 1))

opinionleaders_reddit = opinionleaders_reddit[
    ["rank", SCORE_COL, "comment_time_ams", STABLE_ID_COL, COMMENT_COL]
]

# --- GitHub-ready save path (stable) ---
OUTPUT_DIR = Path("data/processed")
if not OUTPUT_DIR.exists():
    alt = Path("../data/processed")
    if alt.parent.exists():
        OUTPUT_DIR = alt

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

save_path = OUTPUT_DIR / "reddit_opinion_leaders_144.csv"
opinionleaders_reddit.to_csv(save_path, index=False, encoding="utf-8")

print(f"💾 Saved 144 Reddit opinion leaders to:\n{save_path}")

styled_table_reddit = (
    opinionleaders_reddit
        .style
        .set_table_attributes('style="border-collapse:collapse"')
        .hide(axis="index")
        .set_properties(
            subset=[COMMENT_COL],
            **{
                "white-space": "normal",
                "word-wrap": "break-word",
                "width": "770px"
            }
        )
)

styled_table_reddit


**2.1 Thematic Analysis**


This part thematically codes each comment within the subset of opinion leaders. The following categorizations have been identified:

- **Runway Looks & Design Fit:** About the lingerie costumes, styling, hair/makeup, wings, design
- **Model Performance:** About the model's walk, energy, presence on the runway
- **Casting Choices & Nepotism:** Discourse about casting choices with reference to diversity and modelling standards and nepotism
- **Models:** Pictured models in their runway look
- **Artists:** About the artists that performed
- **Self-comparison:** References about oneself
- **Celebrity:** Celebrity references
- **Brand Ambassadors:** Brand ambassadors of VS Collective
- **Cheating scandal:** Regarding a specific incident about one of the models
- **Brand Products:** Experiences, opinions and reviews about brand products
- **Humor:** Sarcasm, irony, jokes, memes
- **Body & Beauty Evaluations:** Analysis, compliments, admiration or critique relating to physical appearance
- **Overall Sentiment:** Overall attitude towards the brand

Below follows the thematic distribution of comments and likecounts, visualised in a pie chart. Furthermore, all comments and their related themes are be provided as well.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


CANDIDATE_LEADERS_PATHS = [
    Path("data/processed/reddit_opinion_leaders_144.csv"),
    Path("../data/processed/reddit_opinion_leaders_144.csv"),
]

leaders_path = next((p for p in CANDIDATE_LEADERS_PATHS if p.exists()), None)
if leaders_path is None:
    raise FileNotFoundError(
        "Could not find reddit_opinion_leaders_144.csv.\n"
        "Run the previous block first to generate it.\n"
        "Expected: data/processed/reddit_opinion_leaders_144.csv"
    )

leaders = pd.read_csv(leaders_path)
print(f"Loaded opinion leader list with {len(leaders):,} rows from:\n{leaders_path}")

# Required columns
REQUIRED = ["rank", "score", "stable_index", "body"]
missing_cols = [c for c in REQUIRED if c not in leaders.columns]
if missing_cols:
    raise ValueError(
        f"Missing required columns in leaders file: {missing_cols}. "
        f"Available columns: {list(leaders.columns)}"
    )

leaders["stable_index"] = pd.to_numeric(leaders["stable_index"], errors="coerce")
leaders = leaders.dropna(subset=["stable_index"]).copy()
leaders["stable_index"] = leaders["stable_index"].astype(int)

leaders["score"] = pd.to_numeric(leaders["score"], errors="coerce").fillna(0).astype(int)
leaders["body"] = leaders["body"].fillna("").astype(str).str.strip()

TOP_N = len(leaders)
print(f"Using TOP_N = {TOP_N} comments keyed by 'stable_index'.")

ranked = leaders.rename(columns={
    "score": "likeCount",
    "body": "comment"
}).copy()

ranked = ranked[["rank", "stable_index", "likeCount", "comment"]]

themes_by_index = {
    "Runway Looks & Design Fit": [
        77, 78, 79, 670, 83, 1205, 99, 629, 1177, 628, 958,
        116, 1274, 1460, 790, 97, 94, 630, 1261, 829, 656, 767,
        895, 1048, 657, 672, 1583, 691, 1027, 833, 109, 999, 1006,
        1279, 1291, 1214, 977, 1037, 794, 632, 655
    ],
    "Self-comparison": [1222, 1471, 1591, 1645, 1472],
    "Models": [
        82, 86, 88, 711, 80, 84, 91, 81, 93, 102, 87, 712, 90, 100, 101,
        765, 104, 92, 96, 120, 105, 98, 89, 855, 702, 106, 917, 95, 126,
        144, 110, 103, 922, 155, 107, 133, 830
    ],
    "Model Performance": [671, 1211, 673, 79, 1278, 125, 731, 925, 1214, 736, 1247, 169, 1212],
    "Casting Choices & Nepotism": [85, 83, 757, 754, 958, 753, 756, 999, 912, 755, 1255, 169],
    "Artists": [142],
    "Cheating Scandal": [1304, 853, 1502, 1620],
    "Brand Ambassadors": [1482],
    "Brand Products": [884, 767, 1311, 1577, 94, 1451],
    "Celebrity": [116, 954, 735, 1343, 1239],
    "Humor": [627, 1169, 1494, 1170, 657, 976, 1250, 631, 1472, 1102],
    "Body & Beauty Evaluations": [
        83, 757, 1177, 116, 737, 792, 769, 767, 895, 738, 1275, 655, 1503,
        115, 768, 1332, 833, 999, 766, 793, 854, 125, 977, 1132, 771
    ],
    "Overall Sentiment": [99, 118, 1184, 1440, 122, 131, 959, 985, 655, 1346]
}

index_to_themes = {}
for theme, indices in themes_by_index.items():
    for idx in indices:
        index_to_themes.setdefault(int(idx), []).append(theme)

def join_labels(idx: int) -> str:
    return ", ".join(index_to_themes.get(idx, []))

ranked["themes"] = ranked["stable_index"].apply(join_labels)

missing = ranked.loc[ranked["themes"].eq(""), "stable_index"].tolist()
if missing:
    print(f"⚠️ stable_index values in top-{TOP_N} with NO theme label:")
    print(sorted(missing))
else:
    print(f"✅ All top-{TOP_N} Reddit comments have at least one theme label.")

ranked["theme_list"] = ranked["themes"].apply(lambda x: x.split(", ") if x else [])
long = ranked.explode("theme_list").rename(columns={"theme_list": "theme"})
long = long[long["theme"] != ""]

counts = long.groupby("theme")["stable_index"].nunique().sort_values(ascending=False)
upvotes = long.groupby("theme")["likeCount"].sum().sort_values(ascending=False)

summary_df = pd.DataFrame({
    "Comments": counts,
    "Upvotes": upvotes,
    "Share of Upvotes (%)": (upvotes / upvotes.sum() * 100).round(1),
}).sort_values("Upvotes", ascending=False)

print("\n📊 THEME SUMMARY\n")
print(summary_df.to_string())

plt.figure(figsize=(10, 7))

wedges, texts, autotexts = plt.pie(
    summary_df["Upvotes"],
    labels=None,
    autopct="%1.1f%%",
    startangle=140,
    pctdistance=0.75
)

plt.title("Thematic Engagement Distribution")

plt.legend(
    wedges,
    summary_df.index,
    title="Theme",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)

plt.tight_layout()
plt.show()


OUTPUT_DIR = Path("data/processed")
if not OUTPUT_DIR.exists():
    alt = Path("../data/processed")
    if alt.parent.exists():
        OUTPUT_DIR = alt
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUTPUT_DIR / "top144_reddit_comments_labeled_by_theme.csv"
ranked.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n📁 Saved labeled top-{TOP_N} Reddit comments to:\n{out_path}")


In [ ]:
import pandas as pd
import textwrap
from pathlib import Path

CANDIDATE_PATHS = [
    Path("data/processed/top144_reddit_comments_labeled_by_theme.csv"),
    Path("../data/processed/top144_reddit_comments_labeled_by_theme.csv"),
]

reddit_labeled_path = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if reddit_labeled_path is None:
    raise FileNotFoundError(
        "Could not find top144_reddit_comments_labeled_by_theme.csv.\n"
        "Run the previous labeling block first.\n"
        "Expected: data/processed/top144_reddit_comments_labeled_by_theme.csv"
    )

df_labeled_comments = pd.read_csv(reddit_labeled_path)
print(f"✅ Loaded labeled Reddit comments from:\n{reddit_labeled_path}")


if "index" in df_labeled_comments.columns:
    df_labeled_comments = df_labeled_comments.rename(columns={"index": "stable_index"})


df_labeled_comments["themes"] = df_labeled_comments["themes"].apply(
    lambda x: [theme.strip() for theme in x.split(",")] if pd.notna(x) and x else []
)

all_unique_themes = sorted(
    {theme for sublist in df_labeled_comments["themes"] for theme in sublist}
)

MAX_WIDTH = 100

print("\nReddit Comments by Theme\n")

for theme in all_unique_themes:
    print(f"\n--- 🔵 Theme: {theme} ---")

    theme_comments = df_labeled_comments[
        df_labeled_comments["themes"].apply(lambda x: theme in x)
    ]

    if theme_comments.empty:
        print(f"  No comments found for '{theme}'.")
        continue

    theme_comments = theme_comments.sort_values(by="likeCount", ascending=False)

    for i, row in enumerate(theme_comments.itertuples(index=False), start=1):
        wrapped_comment = textwrap.fill(
            row.comment,
            width=MAX_WIDTH,
            break_long_words=False,
            replace_whitespace=False,
        )
        print(f"  {i}. 👍 {row.likeCount} upvotes  |  stable_index: {row.stable_index}")
        print(f"     {wrapped_comment}")
        print("-" * MAX_WIDTH)


**3.0 Sentiment Analysis**

This includes a sentiment analysis on the overall dataset, as well as a manual review of 100 comments. Here, the doubtful or incorrect classifications are separated for closer review, before calculating the accuracy rate.  

In [ ]:
!pip install -q transformers torch

from transformers import pipeline
import pandas as pd
import torch
from pathlib import Path

df_reddit_new = pd.read_csv(CLEAN_REDDIT_DATA)
print(f"Comments loaded: {len(df_reddit_new):,}")

comments_df = df_reddit_new.copy()
comments_df["body"] = comments_df["body"].fillna("").astype(str)

MODEL_NAME = "j-hartmann/sentiment-roberta-large-english-3-classes"

use_cuda = torch.cuda.is_available()

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model=MODEL_NAME,
    device=0 if use_cuda else -1,
    torch_dtype=torch.float16 if use_cuda else None,
    truncation=True,
    max_length=512,
    padding=True
)

print("Model id2label:", sentiment_pipeline.model.config.id2label)

def normalize_label(lbl: str) -> str:
    lbl = str(lbl).strip()

    if lbl.upper().startswith("LABEL_"):
        idx = int(lbl.split("_")[-1])
        lbl = sentiment_pipeline.model.config.id2label.get(idx, lbl)

    return str(lbl).upper()

results = []
batch_size = 32

print("\nℕ Running sentiment analysis on entire dataset...\n")

for i in range(0, len(comments_df), batch_size):
    batch_texts = comments_df["body"].iloc[i:i + batch_size].tolist()

    batch_results = sentiment_pipeline(
        batch_texts,
        truncation=True,
        max_length=512,
        padding=True
    )

    results.extend(batch_results)
    print(
        f"Processed {min(i + batch_size, len(comments_df))}/{len(comments_df)} comments",
        end="\r"
    )

print("\n✅ Sentiment analysis complete!")

comments_df["sentiment_label"] = [normalize_label(r["label"]) for r in results]
comments_df["sentiment_score"] = [r["score"] for r in results]

# --- GitHub-ready output path ---
OUTPUT_DIR = Path("data/processed")
if not OUTPUT_DIR.exists():
    alt = Path("../data/processed")
    if alt.parent.exists():
        OUTPUT_DIR = alt
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "sorted_reddit_comments_with_sentiment.csv"
comments_df.to_csv(output_path, index=False, encoding="utf-8")
print(f"\n💾 Enriched dataset saved to: {output_path}")

# Preview
comments_df[["body", "sentiment_label", "sentiment_score"]].head(5)


In [ ]:
import pandas as pd
from pathlib import Path

CANDIDATE_PATHS = [
    Path("data/processed/sorted_reddit_comments_with_sentiment.csv"),
    Path("../data/processed/sorted_reddit_comments_with_sentiment.csv"),
]

REDDIT_SENTIMENT = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if REDDIT_SENTIMENT is None:
    raise FileNotFoundError(
        "Could not find sorted_reddit_comments_with_sentiment.csv.\n"
        "Run the sentiment generation block first.\n"
        "Expected: data/processed/sorted_reddit_comments_with_sentiment.csv"
    )

df_reddit = pd.read_csv(REDDIT_SENTIMENT)
print(f"Loaded {len(df_reddit):,} Reddit rows from: {REDDIT_SENTIMENT}")


In [ ]:
import pandas as pd

df_sentiment = pd.read_csv(REDDIT_SENTIMENT)
pd.set_option("display.max_colwidth", None)
print(f"Loaded {len(df_sentiment):,} rows.")

df_sentiment[["body", "sentiment_label", "sentiment_score"]].head(100)


In [ ]:
pd.set_option("display.max_colwidth", None)
indices = [4, 6, 11, 13, 20, 23, 39, 47, 59, 63, 72, 76]
df_sentiment.loc[indices, ["body", "sentiment_label", "sentiment_score"]]


**Review of Misclassifications**

This dataset includes a lot of detailed critical reviews of the show, sometimes forming very long paragraphs. In some cases comments provided a fairly balanced mixture of positive and negative feedback, where a neutral sentiment aligned better than an either negative or positive sentiment. When comments contained positive language, this was often classified as a positive sentiment, even though it was used to highlight this year’s inadequacy by drawing attention to missing desirable elements. Or comments stating to ‘agree’ with minimal context, are also classified as positive, even though this could be in reference to a negative opinion. These wrong classifications show how the model relies on the meaning of words, but has limited contextual awareness that contributes to the overall expressed sentiment.


*Accuracy report of 88%*

**3.1 Sentiment Popularity**

This reviews the popularity of each sentiment, first by reviewing their overall presence in the dataset, the top comments of each sentiment and the average like count per category.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(REDDIT_SENTIMENT)
print(f"✅ Loaded {len(df):,} Reddit comments with sentiment data")

counts = df["sentiment_label"].value_counts()
percentages = (counts / counts.sum() * 100).round(2)


plt.figure(figsize=(6, 4))
bars = plt.bar(
    counts.index,
    counts.values,
    color=["#E74C3C", "#2ECC71", "#F1C40F"]
)

plt.title("Reddit Sentiment Distribution", fontsize=13, pad=10)
plt.xlabel("Sentiment Category", fontsize=11)
plt.ylabel("Number of Comments", fontsize=11)
plt.xticks(rotation=0)


for bar, pct in zip(bars, percentages):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + counts.max() * 0.01,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()
plt.show()


print("\n📊 Sentiment Breakdown:")
for label, pct in percentages.items():
    print(f"{label}: {pct:.2f}% ({counts[label]} comments)")


In [ ]:
import pandas as pd


df_sentiment = pd.read_csv(REDDIT_SENTIMENT)
df_sentiment.columns = df_sentiment.columns.str.lower()

def get_refined_sentiment(row):
    if (row["sentiment_label"] == "POSITIVE" or row["sentiment_label"] == "NEGATIVE") and row["sentiment_score"] < 0.6:
        return "NEUTRAL"
    else:
        return row["sentiment_label"]

df_sentiment["sentiment_label_refined"] = df_sentiment.apply(get_refined_sentiment, axis=1)

most_negative_upvoted = (
    df_sentiment[df_sentiment["sentiment_label_refined"].str.lower().str.contains("neg")]
    .sort_values("score", ascending=False)
    .head(15)
)

print("\n🔴 Top 15 Most-Upvoted Negative Comments:\n")
for _, row in most_negative_upvoted.iterrows():
    print(f"🖼 {row['body'][:500]}")
    print(f"   🔹 Sentiment: {row['sentiment_label_refined']} ({row['sentiment_score']:.3f}) | Upvotes: {row['score']}\n")


most_positive_upvoted = (
    df_sentiment[df_sentiment["sentiment_label_refined"].str.lower().str.contains("pos")]
    .sort_values("score", ascending=False)
    .head(15)
)

print("\n🟢 Top 15 Most-Upvoted Positive Comments:\n")
for _, row in most_positive_upvoted.iterrows():
    print(f"🖼 {row['body'][:500]}")
    print(f"   🔹 Sentiment: {row['sentiment_label_refined']} ({row['sentiment_score']:.3f}) | Upvotes: {row['score']}\n")


most_neutral_upvoted = (
    df_sentiment[df_sentiment["sentiment_label_refined"].str.lower().str.contains("neu")]
    .sort_values("score", ascending=False)
    .head(15)
)

print("\n🟡 Top 15 Most-Upvoted Neutral Comments:\n")
for _, row in most_neutral_upvoted.iterrows():
    print(f"🖼 {row['body'][:500]}")
    print(f"   🔹 Sentiment: {row['sentiment_label_refined']} ({row['sentiment_score']:.3f}) | Upvotes: {row['score']}\n")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


df = pd.read_csv(REDDIT_SENTIMENT)

df.columns = df.columns.str.lower()


df = df.rename(columns={
    "body": "comment",
    "text": "comment",
    "score": "upvotes"
})


df["upvotes"] = pd.to_numeric(df["upvotes"], errors="coerce").fillna(0)


df["sentiment_label"] = df["sentiment_label"].astype(str).str.lower()


df["sentiment_label_plot"] = df["sentiment_label"].str.title()

sentiment_stats = (
    df.groupby("sentiment_label_plot")["upvotes"]
      .agg(["mean", "sum", "count"])
      .reset_index()
      .sort_values("mean", ascending=False)
)


color_map = {
    "Positive": "#2ECC71",
    "Neutral": "#F1C40F",
    "Negative": "#E74C3C"

}
colors = [color_map.get(label, "gray") for label in sentiment_stats["sentiment_label_plot"]]


plt.figure(figsize=(7, 4))
bars = plt.bar(sentiment_stats["sentiment_label_plot"], sentiment_stats["mean"], color=colors)
plt.title("Average Upvotes per Sentiment Category (Reddit)")
plt.ylabel("Average Upvotes per Comment")
plt.xlabel("Sentiment")


y_pad = sentiment_stats["mean"].max() * 0.01
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + y_pad,
        f"{bar.get_height():.1f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()
plt.show()


print("\n📊 Sentiment Popularity Summary")
for _, row in sentiment_stats.iterrows():
    print(f"{row['sentiment_label_plot']}: {int(row['count'])} comments | "
          f"Avg Upvotes: {row['mean']:.1f} | Total Upvotes: {row['sum']:.0f}")


**4.0 Text Frequency Analysis**

This part includes an analysis of the most frequent words in the dataset. From the most frequent or thematically relevant words, the top 10 most liked comments are reviewed, to gain deeper contextual awareness.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import re

comments_df = pd.read_csv(CLEAN_REDDIT_DATA)

custom_stopwords = {
    's', 't', 'm', 're', 've', 'll', 'd', 'im', 'amp', 'u', 'ur', 'https', 'did',
    'co', 'com', 'www', 'like', 'also', 'one', 'would', 'could', 'get', 'didn',
     'don', 'just'
}


def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


cleaned_comments = comments_df["body"].dropna().astype(str).apply(clean_text)


vectorizer = CountVectorizer(
    stop_words="english",
    token_pattern=r"\b[a-z]{2,}\b"
)


X = vectorizer.fit_transform(cleaned_comments)
words = vectorizer.get_feature_names_out()
counts = X.sum(axis=0).A1

word_counts = pd.DataFrame({"word": words, "count": counts})


word_counts = word_counts[~word_counts["word"].isin(custom_stopwords)]


top_words = (
    word_counts
    .sort_values(by="count", ascending=False)
    .head(15)
    .reset_index(drop=True)
)


top_words.index = top_words.index + 1

print("Top 15 Most Used Words (Reddit):")
display(
    top_words.style.set_properties(
        **{
            "border": "1px solid black",
            "text-align": "left",
        }
    )
)


In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

TOP_N_WORDS = 15
TOP_N_COMMENTS = 15


text_col = "body"

for word in top_words["word"].head(TOP_N_WORDS):
    subset = comments_df[
        comments_df[text_col].str.contains(rf"\b{word}\b", case=False, regex=True)
    ]

    if subset.empty:
        continue

    top_comments = (
        subset.sort_values("score", ascending=False)
              .head(TOP_N_COMMENTS)[[text_col, "score", "stable_index"]]
              .reset_index(drop=True)
    )

    print(f"\n🔹 Top {TOP_N_COMMENTS} most-liked comments containing '{word}':\n")
    display(top_comments)